# Visualizing subsequence distances with an Mplot

An Mplot shows the distances between all pairs of subsequences at a chosen
window size. Each axis gives the start of a subsequence, and the color at
that pair of starts shows how similar the two windows are. Repeating
patterns appear as low-distance regions away from the main diagonal.

This example uses the approach in
[Introducing Mplots: scaling time series recurrence plots to massive datasets](https://doi.org/10.1186/s40537-024-00954-1)
by Shahcheraghi et al. We'll create a short series with three occurrences of
the same waveform and use `stumpy.mass` to compare its windows.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import stumpy

rng = np.random.default_rng(42)
T = rng.normal(0, 0.25, size=512)
m = 48
starts = [55, 220, 400]
phase = np.linspace(0, 2 * np.pi, m, endpoint=False)
waveform = np.sin(phase) + 0.4 * np.sin(3 * phase)

for start, scale, offset in zip(starts, [1.0, 0.6, 1.3], [0.0, 1.0, -0.6]):
    T[start : start + m] = scale * waveform + offset

series_fig, ax = plt.subplots(figsize=(10, 2.6), layout="constrained")
ax.plot(T, linewidth=1)
for start in starts:
    ax.axvspan(start, start + m - 1, color="tab:orange", alpha=0.2)
ax.set(xlabel="Sample", ylabel="Value", title="Series with three repeated waveforms")
plt.show()

## Compare every pair of windows

For each start `i`, `stumpy.mass(T[i : i + m], T)` computes one row of the
Mplot. By default, MASS z-normalizes each window, so the inserted waveforms
remain similar despite their different amplitudes and offsets.
This self-join produces a symmetric matrix, with zeros on the diagonal
where each window is compared with itself.

In [ ]:
n_windows = T.size - m + 1
distances = np.vstack(
    [stumpy.mass(T[i : i + m], T, query_idx=i) for i in range(n_windows)]
)

distance_fig, ax = plt.subplots(figsize=(6.2, 5), layout="constrained")
image = ax.imshow(distances, cmap="viridis", interpolation="nearest", origin="lower")
ax.set(
    xlabel="Target window start",
    ylabel="Query window start",
    title="Mplot of the synthetic time series",
)
distance_fig.colorbar(image, ax=ax, label="Z-normalized Euclidean distance")
plt.show()

## Highlight similar nontrivial pairs

Nearby windows overlap and often look similar for that reason. Mask the
band around the diagonal using the same exclusion zone as STUMPY's default
self-join, which drops pairs whose starts are within `ceil(m / 4)` samples
of each other. Next, show only distances below 1.5. This threshold is
specific to the example. Changing it changes which matches are visible.

In [ ]:
positions = np.arange(n_windows)
exclusion_zone = int(np.ceil(m / 4))
overlap = np.abs(positions[:, None] - positions) <= exclusion_zone
nontrivial_distances = np.where(overlap, np.inf, distances)
similar = nontrivial_distances < 1.5

matches_fig, ax = plt.subplots(figsize=(6, 5), layout="constrained")
ax.imshow(similar, cmap="Greys", interpolation="nearest", origin="lower")
starts = np.asarray(starts)
query, target = np.nonzero(similar[np.ix_(starts, starts)])
ax.scatter(
    starts[target], starts[query], s=85, facecolors="none", edgecolors="tab:orange"
)
ax.set(
    xlabel="Target window start",
    ylabel="Query window start",
    title="Nontrivial pairs with distance below 1.5",
)
plt.show()

The six orange circles mark the three known matches and their transposes.
The diagonal is empty because self-matches are excluded.

Each column's smallest remaining distance is the corresponding matrix
profile value. We can verify this by comparing the column minima with
`stumpy.stump`.

In [ ]:
profile_from_mplot = nontrivial_distances.min(axis=0)
profile_from_stump = stumpy.stump(T, m).P_
np.testing.assert_allclose(profile_from_mplot, profile_from_stump, atol=1e-6)

## Compare two spacecraft telemetry traces

[Figure 19 of the paper](https://doi.org/10.1186/s40537-024-00954-1) compares channel
G-1's test trace with the first 500 training samples at window length 80.
In this AB-join, each row is a training window and each column is a
test window. The two traces are separate, so no exclusion zone is needed.

We use the first column of the [Telemanom archive](https://github.com/khundman/telemanom),
which holds the telemetry readings. The paper calls the channel MSL G1
and marks samples 4797–4871. The [published Telemanom labels](https://github.com/khundman/telemanom/blob/master/labeled_anomalies.csv)
list G-1 under SMAP and mark 4770–4890. The shaded region uses the
Telemanom interval.

In [ ]:
from io import BytesIO
from urllib.request import urlopen

archive = (
    "https://huggingface.co/datasets/appleparan/telemanom/resolve/"
    "2d22e1061be83a88b7b9e48df35163d5147adc9d/data/data"
)
with urlopen(f"{archive}/train/G-1.npy", timeout=30) as response:
    train = np.load(BytesIO(response.read()), allow_pickle=False)[:500, 0]
with urlopen(f"{archive}/test/G-1.npy", timeout=30) as response:
    test = np.load(BytesIO(response.read()), allow_pickle=False)[:, 0]

telemetry_m = 80
telemetry_distances = np.vstack(
    [
        stumpy.mass(train[i : i + telemetry_m], test)
        for i in range(train.size - telemetry_m + 1)
    ]
)

In [ ]:
telemetry_fig, (signal_ax, plot_ax) = plt.subplots(
    2, 1, figsize=(10, 5), sharex=True, layout="constrained",
    height_ratios=[1, 2],
)
signal_ax.plot(test, linewidth=1)
signal_ax.set(ylabel="Telemetry value", title="G-1 test trace near the labeled anomaly")
image = plot_ax.imshow(
    telemetry_distances, cmap="viridis", interpolation="nearest",
    aspect="auto", origin="lower",
)
plot_ax.set(
    xlabel="Test sample or window start", ylabel="Training window start",
    xlim=(4400, 5200),
)
for ax in (signal_ax, plot_ax):
    ax.axvspan(4770, 4890, color="tab:orange", alpha=0.25)
telemetry_fig.colorbar(image, ax=[signal_ax, plot_ax], label="Z-normalized distance")
plt.show()

The repeating waves produce diagonal bands of low distance. The bands
weaken where the test trace flattens inside the shaded interval.

## Memory use

An Mplot stores one distance for every pair of windows, so its memory
use grows quadratically with series length. When you need only the nearest
neighbor of each window, compute the matrix profile directly with
`stumpy.stump`. For a large Mplot, see the paper's discussion of
downsampling and selectively computing parts of the matrix.